In [1]:
# imports and settings

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import utils as ut

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

Imports complete.
Settings: height=800, width=1400, font_size=16
Imports complete.


In [2]:
# BG Noise WAVs
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(folder_path, file))
    data.append(_data)
data = np.concatenate(data)
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")


data final shape: (6615000,) of duration: 300.0 seconds


In [3]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 8000
norm_size = 5
s2g_quantization_levels = 3
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

welch_detector = ut.WelchDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)

In [4]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_welch_scores = []
f0_2_welch_scores = []

f0_1_td_scores = []
f0_2_td_scores = []

f0_1_s2g_scores = []
f0_2_s2g_scores = []

f0_1 = 631  # Hz
f0_2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    P = welch_detector.get_score_1fb(_data, f0_1)
    f0_1_welch_scores.append(P)
    P = welch_detector.get_score_1fb(_data, f0_2)
    f0_2_welch_scores.append(P)

    TD = talmon_detector.get_score_1fb(_data, f0_1)
    f0_1_td_scores.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_2)
    f0_2_td_scores.append(TD)

    K = s2g_detector.get_score_1fb(_data, f0_1)
    f0_1_s2g_scores.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_2)
    f0_2_s2g_scores.append(K)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")

completed all detectors


In [6]:
fig = make_subplots(rows=3, 
                    cols=1, 
                    row_heights=[0.7, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "distance", "Talmon", "S2G"])

fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Talmon f={f0_1} Hz', marker_color='blue', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Talmon f={f0_2} Hz', marker_color='green', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=3, col=1)


fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Score", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)

fig.update_layout(height=800, width=1200, title_text=f"Only noise, scores for f0 = {f0_1}, {f0_2} Hz")
fig.show(renderer="browser")
fig.write_image("../results/plots/nb89_noise_only.pdf")